In [11]:
from pathlib import Path

from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough


## Document loading

In [5]:
data_path = Path('/Users/rajshree/Aashish/Code/Github/Automotive_RAG/data/processed/braking_system_requirements.md')

data = data_path.read_text(encoding="utf-8")
type(data)

str

In [6]:
data

'Braking System Requirements Specification\n\n# 1. Scope\n\nThis document specifies synthetic but realistic requirements for automotive braking systems, covering both legacy hydraulic architectures and modern brake-by-wire one-box electro- hydraulic architectures. The scope includes service braking, parking braking, ABS, ESC, EPB, regenerative braking coordination, diagnostics, degraded modes, and functional safety requirements aligned with common automotive braking and safety practices.[web:6][web:7]\n\n[web:17][web:22] The document is written as a pure requirements specification. It excludes RAG ingestion guidance, chunking strategy, retrieval examples, and corpus-structuring content that is not part of the braking system specification itself.[file:32][file:33]\n\n# 2. System Overview\n\nA legacy hydraulic brake system uses a brake pedal, booster, master cylinder, hydraulic lines, and wheel-end friction brakes to convert driver input into braking torque. ABS and ESC functions are typ

## Chunking

In [7]:
Headers_to_split = [
    ('#', 'H1'),
    ('##', 'H2'),
    ('###', 'H3')
]

In [16]:
splitter = MarkdownHeaderTextSplitter(headers_to_split_on = Headers_to_split,
                                strip_headers = False)
            
splitter

In [21]:
header_splits = splitter.split_text(data)
header_splits

[Document(metadata={}, page_content='Braking System Requirements Specification'),
 Document(metadata={'H1': '1. Scope'}, page_content='# 1. Scope  \nThis document specifies synthetic but realistic requirements for automotive braking systems, covering both legacy hydraulic architectures and modern brake-by-wire one-box electro- hydraulic architectures. The scope includes service braking, parking braking, ABS, ESC, EPB, regenerative braking coordination, diagnostics, degraded modes, and functional safety requirements aligned with common automotive braking and safety practices.[web:6][web:7]  \n[web:17][web:22] The document is written as a pure requirements specification. It excludes RAG ingestion guidance, chunking strategy, retrieval examples, and corpus-structuring content that is not part of the braking system specification itself.[file:32][file:33]'),
 Document(metadata={'H1': '2. System Overview'}, page_content='# 2. System Overview  \nA legacy hydraulic brake system uses a brake pe

In [22]:
len(header_splits)

71

In [23]:
header_splits[5].page_content

'# 4. Definitions  \nService Brake: Primary braking function used to decelerate or stop the vehicle during driving.[web:7]  \nParking Brake: Function intended to hold the vehicle stationary under specified loading and gradient conditions.[web:7]  \nABS: Anti-lock Braking System used to prevent wheel lock during braking.[web:6][web:8]  \nESC/ESP: Electronic Stability Control function used to stabilize vehicle yaw behavior through selective braking interventions.[web:30][web:6]  \nEPB: Electric Parking Brake.[web:18][web:28]  \nBBW: Brake-by-wire braking architecture using electrical/electronic interpretation of brake demand and electronically commanded brake actuation.[web:23][web:28]  \nEHB: Electro-hydraulic braking implementation of brake-by-wire.[web:18][web:28]  \nHARA: Hazard Analysis and Risk Assessment within the ISO 26262 safety lifecycle.[web:17]  \n[web:22] ASIL: Automotive Safety Integrity Level defined by ISO 26262.[web:17][web:22]'

In [24]:
chunk_size = 500
chunk_overlap = 50
text_splitter = RecursiveCharacterTextSplitter(chunk_size = chunk_size, 
                                                chunk_overlap = chunk_overlap)

text_splits = text_splitter.split_documents(header_splits)
text_splits                                                

[Document(metadata={}, page_content='Braking System Requirements Specification'),
 Document(metadata={'H1': '1. Scope'}, page_content='# 1. Scope  \nThis document specifies synthetic but realistic requirements for automotive braking systems, covering both legacy hydraulic architectures and modern brake-by-wire one-box electro- hydraulic architectures. The scope includes service braking, parking braking, ABS, ESC, EPB, regenerative braking coordination, diagnostics, degraded modes, and functional safety requirements aligned with common automotive braking and safety practices.[web:6][web:7]'),
 Document(metadata={'H1': '1. Scope'}, page_content='[web:17][web:22] The document is written as a pure requirements specification. It excludes RAG ingestion guidance, chunking strategy, retrieval examples, and corpus-structuring content that is not part of the braking system specification itself.[file:32][file:33]'),
 Document(metadata={'H1': '2. System Overview'}, page_content='# 2. System Overvi

In [25]:
len(text_splits)

79

## Embedding

In [26]:
embedding_model = "nomic-embed-text"

embeddings = OllamaEmbeddings(model = embedding_model)
embeddings

OllamaEmbeddings(model='nomic-embed-text', dimensions=None, validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

In [27]:
demo_embed_vector = embeddings.embed_query('What is this document about?')
demo_embed_vector

[0.0025288067,
 0.029839395,
 -0.14158046,
 -0.059699185,
 0.0059005264,
 -0.0068343463,
 0.007591919,
 0.022525791,
 -0.03289859,
 -0.017683048,
 0.0016295422,
 0.060233433,
 0.088059776,
 -0.038054384,
 0.07279688,
 -0.01645569,
 -0.04937463,
 -0.024461718,
 0.0017591813,
 0.03968412,
 -0.013947343,
 0.03330593,
 -0.093892135,
 0.026083,
 0.10914662,
 0.05979161,
 0.0050064013,
 0.006554505,
 -0.06324409,
 -0.064529985,
 0.0055512073,
 0.00053854944,
 0.030519847,
 0.0015537417,
 -0.03241388,
 -0.085694805,
 0.06646082,
 0.0105783865,
 -0.018122124,
 0.0050073303,
 -0.039068144,
 -0.009041463,
 -0.05164462,
 -0.08085197,
 0.011571098,
 0.029759895,
 0.026463246,
 -0.023136133,
 -0.008556344,
 -0.053670954,
 0.034820598,
 -0.0430573,
 0.010678885,
 -0.0051234807,
 0.040995803,
 0.0070663304,
 -0.033040788,
 -0.0026261457,
 -0.035993967,
 -0.008921344,
 0.10002578,
 0.1001085,
 -0.055081006,
 0.08276131,
 0.032483917,
 -0.06783348,
 -0.02106628,
 -0.023241062,
 0.0027577782,
 -0.024600

In [28]:
len(demo_embed_vector)

768

## Vector Store

In [30]:
persist_dir = '/Users/rajshree/Aashish/Code/Github/Automotive_RAG/data/vectorstore'

In [33]:
vector_store = Chroma.from_documents(documents = text_splits,
                                    embedding= embeddings,
                                    persist_directory=persist_dir)

vector_store                                    

In [38]:
demo_search = vector_store.similarity_search('Pressure control in ABS', k = 3)

demo_search

[Document(id='e4afcb63-5bd5-4db7-aa54-b68a530c7bcc', metadata={'H3': 'REQ-ABS-002 – Anti-Lock Pressure Modulation', 'H1': '7. ABS and ESC Requirements'}, page_content='### REQ-ABS-002 – Anti-Lock Pressure Modulation  \nWhen wheel slip exceeds control thresholds, the ABS function shall modulate individual wheel brake pressure to reduce lock-up risk and preserve steerability.[web:6][web:30]'),
 Document(id='48a29341-41b8-45d0-aebb-544a9dda0961', metadata={'H3': 'REQ-ABS-002 – Anti-Lock Pressure Modulation', 'H1': '7. ABS and ESC Requirements'}, page_content='### REQ-ABS-002 – Anti-Lock Pressure Modulation  \nWhen wheel slip exceeds control thresholds, the ABS function shall modulate individual wheel brake pressure to reduce lock-up risk and preserve steerability.[web:6][web:30]'),
 Document(id='ac9c8f19-eeb9-49f9-869a-c7da543c3141', metadata={'H3': 'REQ-ABS-002 – Anti-Lock Pressure Modulation', 'H1': '7. ABS and ESC Requirements'}, page_content='### REQ-ABS-002 – Anti-Lock Pressure Modul

## Chain building and Retrieval

In [39]:
retriever = vector_store.as_retriever(search_type = 'similarity', search_kwargs = {'k': 4})

In [49]:
system_message = """
You are an experienced Automotive engineer working with braking systems for Tier 1 OEMS like VW, BMW.
Answer the provided questions based on the provided context. Do not make up answers if you are not sure about answers from the context. Just
say I don't know about it. Cite the REQ-ID in the response
"""

human_message = """
Context: {context}
Question: {question}
"""

In [50]:
prompt = ChatPromptTemplate.from_messages([
        ('system', system_message),
        ('human', human_message)
])

prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template="\nYou are an experienced Automotive engineer working with braking systems for Tier 1 OEMS like VW, BMW.\nAnswer the provided questions based on the provided context. Do not make up answers if you are not sure about answers from the context. Just\nsay I don't know about it. Cite the REQ-ID in the response\n"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nContext: {context}\nQuestion: {question}\n'), additional_kwargs={})])

In [51]:
def format_text(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

In [52]:
llm = ChatOllama(model = 'llama3.1:8b')

In [53]:
rag_chain = ({
    'context': retriever | format_text,
    'question' : RunnablePassthrough()
}
| prompt
| llm
| StrOutputParser()
)

rag_chain

{
  context: VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x113cc99d0>, search_kwargs={'k': 4})
           | RunnableLambda(format_text),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template="\nYou are an experienced Automotive engineer working with braking systems for Tier 1 OEMS like VW, BMW.\nAnswer the provided questions based on the provided context. Do not make up answers if you are not sure about answers from the context. Just\nsay I don't know about it. Cite the REQ-ID in the response\n"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nContext: {context}\nQuestion: {question}\n'), addit

In [54]:
question = 'What are primary elements of legacy brake system'


In [55]:
rag_chain.invoke(question)

'According to the provided context (REQ-ID: # 3. Architectural Description), the primary elements of a legacy hydraulic brake system are:\n\n1. Brake pedal and mechanical linkage\n2. Vacuum or hydraulic brake booster\n3. Tandem master cylinder\n4. Dual hydraulic circuits\n5. Wheel-end disc or drum brake actuators\n6. Wheel speed sensing and hydraulic modulation components for ABS/ESC-equipped variants.\n\nThese primary elements comprise the legacy architecture of a hydraulic brake system.'

In [56]:
question = 'Can you compare legacy brake system and brake by wire systems?'
rag_chain.invoke(question)


"Based on the context provided, here's a comparison between legacy hydraulic brake systems and Brake-by-Wire (BBW) one-box systems:\n\n**Legacy Hydraulic Brake System:**\n\n* Uses a direct mechanical/hydraulic pedal-to-wheel path\n* Converts driver input into braking torque through a brake pedal, booster, master cylinder, hydraulic lines, and wheel-end friction brakes\n* ABS and ESC functions are typically implemented through an ECU and hydraulic modulation components connected into the hydraulic circuit\n\n**Brake-by-Wire (BBW) One-Box System:**\n\n* Replaces or decouples the direct mechanical/hydraulic pedal-to-wheel path\n* Interprets driver demand through sensors, electronic control logic, and an integrated electro-hydraulic actuator unit\n* Integrates multiple functions, including service braking, ABS, ESC, traction control, and coordination with regenerative braking in electrified vehicles\n\nKey differences:\n\n1. Mechanical vs. Electronic: Legacy systems use a direct mechanical

In [58]:
for chunk in rag_chain.stream(question):
    print(chunk, end = '', flush = True)

Based on the provided context, here's a comparison between legacy hydraulic brake systems and brake-by-wire (BBW) one-box systems:

**Legacy Hydraulic Brake System**

* Uses a direct mechanical/hydraulic pedal-to-wheel path
* Components: brake pedal, booster, master cylinder, hydraulic lines, wheel-end friction brakes
* ABS and ESC functions are typically implemented through an ECU and hydraulic modulation components connected into the hydraulic circuit

**Brake-by-Wire (BBW) One-Box System**

* Replaces or decouples the direct mechanical/hydraulic pedal-to-wheel path
* Interprets driver demand through sensors, electronic control logic, and an integrated electro-hydraulic actuator unit
* Integrates service braking, ABS, ESC, traction control, and coordination with regenerative braking in electrified vehicles

Key differences:

* Legacy system uses a hydraulic circuit, while BBW system is electronic and decouples the pedal-to-wheel path.
* BBW system integrates multiple functions (servi

In [59]:
question = 'What is residual braking? What is its use?'
for chunk in rag_chain.stream(question):
    print(chunk, end = '', flush = True)

Based on the context, I can answer:

Residual Braking refers to the capability of a braking system to provide some level of braking even after a single failure in the service brake path.

Its use is to prevent total loss of braking capability in case of a single-point failure, ensuring that the vehicle can still come to a stop safely.